# Step 4 – Integrating DHNx network losses into the oemof model
**MJ2505 Project 2 – District heating network optimisation**

| | |
|---|---|
| **Goal** | Make the generation system also cover the **distribution losses** of the Step 2 network: $Q_{total}(t) = Q_{demand}(t) + Q_{losses}(t)$. Same five units, same model, same topology – only the demand changes. |
| **Reads** | `results/step2/network_losses.csv` (DHNx) · `data/step3/*` (oemof inputs) · `results/step3/unit_summary.csv` |
| **Writes** | `data/step4/oemof_input_with_losses.csv` (**adjusted load profile**, reused by Step 5) · `results/step4/` |
| **Grading** | B – show explicitly how DHNx losses are transferred, verify units and time resolution, re-solve, quantify changes (generation, dispatch, capacity use, cost, efficiency), identify newly binding capacities |

Legend: 🔧 ASSUMPTION · ✏️ TODO

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from helpers import (DATA, RESULTS, SOLVER, INTEREST_RATE, build_and_solve_heat_system,
                     summarize_units, heat_dispatch, plot_dispatch, UNIT_COLORS)

IN = DATA / "step4"
OUT = RESULTS / "step4"
IN.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Diagnostic (from the course "Prerequisites" document): which Python / packages / solver are used?
import sys, shutil
import pandas as pd
import oemof.solph as solph
import dhnx
from helpers import SOLVER

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("DHNx version:     ", dhnx.__version__)
print("oemof.solph:      ", solph.__version__)
print("pandas:           ", pd.__version__)
print("Solver used:      ", SOLVER, "->", shutil.which("cbc") or "HiGHS via the 'highspy' pip package")

## 1. Load the Step 3 model inputs and the Step 2 network losses

In [ ]:
data = pd.read_csv(DATA / "step3" / "oemof_input.csv", index_col="date", parse_dates=True).asfreq("h")
tech = pd.read_csv(DATA / "step3" / "technology_parameters.csv", index_col="unit")
carriers = pd.read_csv(DATA / "step3" / "carriers.csv", index_col="carrier")
network_losses = pd.read_csv(RESULTS / "step2" / "network_losses.csv")
summary_step3 = pd.read_csv(RESULTS / "step3" / "unit_summary.csv", index_col=0)
network_losses

## 2. Transfer the DHNx losses into the oemof demand (show this explicitly in the report)

**What DHNx gives:** the heat loss of every built pipe, `losses` [kW] = length [m] × `l_factor_fix` [kW/m]. We modelled losses as **independent of the flow** (`l_factor = 0`) because heat loss is driven by the temperature difference between the water and the ground, not by the mass flow. Hence the network loses the **same power in every hour**:

$$\dot Q_{loss}(t) = \sum_{pipes} L_e\, l^{fix}_k \quad [\text{kW}] \;\;\Rightarrow\;\; Q_{loss,year} = \dot Q_{loss} \cdot 8760\,\text{h}$$

**Unit and time-resolution checks**
- DHNx: kW → oemof: MW (÷ 1000).
- Both series hourly with 8760 values; in an hourly model 1 MW for 1 h = 1 MWh.
- The annual loss computed from the hourly profile must equal the DHNx annual value.

✏️ **Optional refinement (good for the discussion / sensitivity):** losses scale with $(T_{supply}+T_{return})/2 - T_{ground}$; a monthly ground-temperature or supply-temperature curve would give a seasonal loss profile.

In [ ]:
Q_loss_kW = network_losses.loc[0, "total_loss_kW"]                  # kW, from DHNx (Step 2)
Q_loss_MW = Q_loss_kW / 1000                                        # MW, oemof unit

data["network losses"] = Q_loss_MW                                  # constant hourly profile [MW]
data["heat demand incl. losses"] = data["heat demand"] + data["network losses"]   # Q_total = Q_demand + Q_losses

# ---- checks: units & time resolution ----
assert len(data) == 8760 and (data.index.to_series().diff().dropna() == pd.Timedelta("1h")).all(), "not hourly 8760"
assert np.isclose(data["network losses"].sum(), network_losses.loc[0, "annual_loss_MWh"]), "annual loss mismatch"
print(f"Network loss: {Q_loss_kW:.2f} kW = {Q_loss_MW:.5f} MW in every hour")
print(f"Annual: demand {data['heat demand'].sum():,.1f} MWh + losses {data['network losses'].sum():,.1f} MWh "
      f"= {data['heat demand incl. losses'].sum():,.1f} MWh "
      f"(losses = {100*data['network losses'].sum()/data['heat demand'].sum():.2f} % of demand)")
print(f"Peak:   {data['heat demand'].max():.4f} MW -> {data['heat demand incl. losses'].max():.4f} MW")

data.to_csv(IN / "oemof_input_with_losses.csv")
print("✅ Adjusted load profile saved to", IN / "oemof_input_with_losses.csv")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 3.5), sharey=True)
for ax, (start, end, title) in zip(axs, [("2019-01-14", "2019-01-20", "Winter week"), ("2019-07-15", "2019-07-21", "Summer week")]):
    d = data[start:end]
    ax.plot(d.index, d["heat demand"], label="consumer demand")
    ax.plot(d.index, d["heat demand incl. losses"], label="demand + network losses")
    ax.set_title(title); ax.set_ylabel("MW"); ax.legend()
plt.tight_layout(); plt.savefig(OUT / "demand_with_losses.png", dpi=150); plt.show()

## 3. Re-solve the same oemof model with the adjusted demand
Same function, technologies and carriers as Step 3 – the only change is `demand_col="heat demand incl. losses"`.

In [ ]:
model, results, parts = build_and_solve_heat_system(data, tech, carriers, demand_col="heat demand incl. losses",
                                                    i=INTEREST_RATE)
sr = model.solver_results
print("Termination condition:", sr.solver.termination_condition if hasattr(sr, "solver")   # CBC
      else sr.get("termination_condition", "n/a"))                                            # HiGHS

dispatch = heat_dispatch(results, parts)
units = [c for c in dispatch.columns if c != "demand"]
assert (dispatch[units].sum(axis=1) - data["heat demand incl. losses"]).abs().max() < 1e-4
print("✅ Demand + losses met in all hours")

summary = summarize_units(results, parts, tech, carriers)
summary.round(1)

## 4. Compare with Step 3 (without losses)
Quantify the change in generation, dispatch, capacities, capacity use and cost per unit.

In [ ]:
cols = ["capacity [MW]", "heat [MWh]", "share of heat [%]", "full-load hours [h]",
        "hours at max capacity [h]", "input energy [MWh]", "total cost [EUR]"]
comparison = pd.concat({"Step 3": summary_step3[cols], "Step 4": summary[cols]}, axis=1)
for c in ["capacity [MW]", "heat [MWh]", "hours at max capacity [h]", "total cost [EUR]"]:
    comparison[("Δ", c)] = summary[c] - summary_step3[c]
comparison.round(2)

### 4.1 Newly binding capacities and capacity utilisation
- **Investment limit binding:** the optimiser built the maximum allowed capacity (`max_capacity_MW`).
- **Operating at maximum:** hours in which a unit runs at its installed capacity (from the table above).

In [ ]:
max_cap = tech["max_capacity_MW"]
binding = pd.DataFrame({
    "max allowed [MW]": max_cap,
    "Step 3 capacity [MW]": summary_step3.loc[tech.index, "capacity [MW]"],
    "Step 4 capacity [MW]": summary.loc[tech.index, "capacity [MW]"],
})
binding["limit binding in Step 3"] = binding["Step 3 capacity [MW]"] >= 0.999 * max_cap
binding["limit binding in Step 4"] = binding["Step 4 capacity [MW]"] >= 0.999 * max_cap
binding["newly binding"] = binding["limit binding in Step 4"] & ~binding["limit binding in Step 3"]
binding

### 4.2 Optional: keep the Step 3 capacities fixed
Answers *"Do units now operate closer to their capacity limits?"* in the strict sense: the Step 3 plant park must now also cover the losses (capacities may not grow). If this run is infeasible, the Step 3 plant could not have supplied the losses.

In [ ]:
tech_fixed = tech.copy()
tech_fixed["max_capacity_MW"] = summary_step3.loc[tech.index, "capacity [MW]"] + 1e-6
try:
    model_f, results_f, parts_f = build_and_solve_heat_system(data, tech_fixed, carriers,
                                                              demand_col="heat demand incl. losses", i=INTEREST_RATE)
except RuntimeError:
    # Expected when Step 3 sized the plant exactly to the peak: the extra losses cannot be covered.
    print("Infeasible: the Step 3 capacities cannot cover demand + losses in the peak hour(s).")
    print("-> The losses make at least one capacity limit binding; Step 4 must invest more (see section 4.1).")
    summary_fixed = None
else:
    summary_fixed = summarize_units(results_f, parts_f, tech_fixed, carriers)
    display(pd.DataFrame({"hours at max – Step 3": summary_step3["hours at max capacity [h]"],
                          "hours at max – Step 4, Step-3 capacities": summary_fixed["hours at max capacity [h]"],
                          "full-load hours – Step 3": summary_step3["full-load hours [h]"],
                          "full-load hours – Step 4, Step-3 capacities": summary_fixed["full-load hours [h]"]}).round(0))

## 5. System-level effects: total generation, cost, efficiency
- **Distribution efficiency** = heat delivered to consumers / heat generated.
- **Fuel/electricity per MWh delivered** shows how losses raise the primary input.

In [ ]:
def system_kpis(summary, delivered):
    gen = summary.loc["TOTAL", "heat [MWh]"]
    return pd.Series({
        "heat generated [MWh]": gen,
        "heat delivered to consumers [MWh]": delivered,
        "distribution efficiency [%]": 100 * delivered / gen,
        "input energy [MWh]": summary.loc["TOTAL", "input energy [MWh]"],
        "input per MWh delivered [-]": summary.loc["TOTAL", "input energy [MWh]"] / delivered,
        "total capacity [MW]": summary.loc["TOTAL", "capacity [MW]"],
        "total annual cost [EUR]": summary.loc["TOTAL", "total cost [EUR]"],
        "cost per MWh delivered [EUR/MWh]": summary.loc["TOTAL", "total cost [EUR]"] / delivered,
    })

kpis = pd.DataFrame({"Step 3 (no losses)": system_kpis(summary_step3, data["heat demand"].sum()),
                     "Step 4 (with losses)": system_kpis(summary, data["heat demand"].sum())})
kpis["change [%]"] = 100 * (kpis.iloc[:, 1] / kpis.iloc[:, 0] - 1)
kpis.round(3)

## 6. Dispatch with losses and the change per unit

In [ ]:
plot_dispatch(dispatch, "Heat production by unit – Step 4 (demand + network losses)")
plt.savefig(OUT / "dispatch.png", dpi=150); plt.show()

dispatch_step3 = pd.read_csv(RESULTS / "step3" / "dispatch.csv", index_col=0, parse_dates=True)
delta = (dispatch[units] - dispatch_step3[units]).resample("ME").sum()
delta.plot.bar(stacked=True, figsize=(12, 4), color=[UNIT_COLORS[u] for u in units],
               title="Change in monthly heat production vs. Step 3 (MWh) – who covers the losses?")
plt.ylabel("MWh"); plt.tight_layout(); plt.savefig(OUT / "dispatch_change.png", dpi=150); plt.show()

## 7. Save results (used by Step 5)

In [ ]:
summary.to_csv(OUT / "unit_summary.csv")
dispatch.to_csv(OUT / "dispatch.csv")
comparison.to_csv(OUT / "comparison_with_step3.csv")
kpis.to_csv(OUT / "system_kpis.csv")
print("Saved to", OUT)

## 8. Questions to answer
✏️

1. **How does including distribution losses change the operation of each generating unit?** *(section 4, 6 – which unit covers the extra MWh and why: it is the marginal unit in most hours)*
2. **Do any units now operate closer to their capacity limits?** *(4.1, 4.2)*
3. **How are total heat generation, operating cost and the generation mix affected?** *(section 5)*
4. Describe the DHNx → oemof link step by step (section 2), incl. the unit/time-resolution checks, and discuss the constant-loss assumption.